# INSTRUCTION FINE-TUNING

## STEP 1: PREPARING DATASET

<div class="alert alert-block alert-success">

In this section, we download and format the instruction dataset for instruction finetuning a
pretrained LLM in this chapter. The dataset consists of 1100 instruction-response pairs.

The following code implements and executes a function to download this dataset, which
is a relatively small file, only 204 KB in size, in JSON format. JSON, or JavaScript Object
Notation, mirrors the structure of Python dictionaries, providing a simple structure for data
interchange that is both human-readable and machine-friendly.

</div>

In [9]:
import json
import os
import urllib
import ssl


In [10]:
def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        print(f"Downloading file from {url}...")
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        print(f"Loading file from {file_path}...")
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

In [11]:
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Loading file from instruction-data.json...
Number of entries: 1100


<div class="alert alert-block alert-success">

The data list , which we loaded from the JSON file contains the 1100 entries of the
instruction dataset. 

Let's print one of the entries to see how each entry is structured:

</div>

In [12]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [13]:
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


### CONVERTING INSTRUCTIONS INTO ALPACA FORMAT

#### What is Alpaca, really?
Alpaca is a Stanford research project that fine-tuned the original LLaMA-7B on ~52k instruction–response pairs generated via a larger model. The appeal was its simplicity and cost efficiency. Key points:

- Base model: Originally LLaMA-7B (older architecture and license constraints). Many modern reproductions use Llama 2/3 or open-llama variants.
- Data: Short, diverse instructions. Great for general instruction-following; limited on complex reasoning.
- Method: LoRA adapters on top of the base with a simple prompt template.
- Pros: Extremely accessible recipe; easy to replicate; runs on commodity GPUs.
- Cons: Results depend heavily on the base model; older Alpaca stacks may lag in safety, reasoning, and license suitability for commercial use. Check base-model terms.

### What is Phi-3?
Phi-3 is Microsoft’s small language model family, engineered to be compact but strong at reasoning and instruction following. It’s trained on high-quality, curated and synthetic data emphasizing correctness, explanations, and alignment. Highlights:

- Sizes: Multiple sizes (e.g., “mini” class around a few billion parameters). Good fit for edge and low-latency server inference.
- Quality focus: Emphasis on textbook-quality and safety-aware data, yielding robust out-of-the-box behavior.
- Efficiency: Strong accuracy-per-parameter and low memory footprint; ideal for QLoRA fine-tunes.
- Availability: Offered through common hubs and cloud catalogs. Review model-specific licensing and usage terms for your deployment context.

In [14]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

<div class="alert alert-block alert-info">
    
This format_input function takes a dictionary entry as input and constructs a formatted
string.

</div>

<div class="alert alert-block alert-success">

 Let's test it to dataset entry data[50], which to looked at earlier:

</div>

In [15]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


### SPLITTING DATASET INTO TRAIN-TEST-VALIDATION

In [16]:
train_portion = int(len(data) * 0.85) # 85% for training
test_portion = int(len(data) * 0.10) # 10% for testing
val_portion = len(data) - train_portion - test_portion # Remaining for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [17]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


<div class="alert alert-block alert-warning">

Having successfully downloaded and partitioned the dataset, and gained a clear
understanding of the dataset prompt formatting, we are now ready for the core
implementation of the instruction finetuning process.

</div>